In [1]:
#****************************************************************************
# (C) Cloudera, Inc. 2020-2023
#  All rights reserved.
#
#  Applicable Open Source License: GNU Affero General Public License v3.0
#
#  NOTE: Cloudera open source products are modular software products
#  made up of hundreds of individual components, each of which was
#  individually copyrighted.  Each Cloudera open source product is a
#  collective work under U.S. Copyright Law. Your license to use the
#  collective work is as provided in your written agreement with
#  Cloudera.  Used apart from the collective work, this file is
#  licensed for your use pursuant to the open source license
#  identified above.
#
#  This code is provided to you pursuant a written agreement with
#  (i) Cloudera, Inc. or (ii) a third-party authorized to distribute
#  this code. If you do not have a written agreement with Cloudera nor
#  with an authorized and properly licensed third party, you do not
#  have any rights to access nor to use this code.
#
#  Absent a written agreement with Cloudera, Inc. (“Cloudera”) to the
#  contrary, A) CLOUDERA PROVIDES THIS CODE TO YOU WITHOUT WARRANTIES OF ANY
#  KIND; (B) CLOUDERA DISCLAIMS ANY AND ALL EXPRESS AND IMPLIED
#  WARRANTIES WITH RESPECT TO THIS CODE, INCLUDING BUT NOT LIMITED TO
#  IMPLIED WARRANTIES OF TITLE, NON-INFRINGEMENT, MERCHANTABILITY AND
#  FITNESS FOR A PARTICULAR PURPOSE; (C) CLOUDERA IS NOT LIABLE TO YOU,
#  AND WILL NOT DEFEND, INDEMNIFY, NOR HOLD YOU HARMLESS FOR ANY CLAIMS
#  ARISING FROM OR RELATED TO THE CODE; AND (D)WITH RESPECT TO YOUR EXERCISE
#  OF ANY RIGHTS GRANTED TO YOU FOR THE CODE, CLOUDERA IS NOT LIABLE FOR ANY
#  DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, PUNITIVE OR
#  CONSEQUENTIAL DAMAGES INCLUDING, BUT NOT LIMITED TO, DAMAGES
#  RELATED TO LOST REVENUE, LOST PROFITS, LOSS OF INCOME, LOSS OF
#  BUSINESS ADVANTAGE OR UNAVAILABILITY, OR LOSS OR CORRUPTION OF
#  DATA.
#
# #  Author(s): Paul de Fusco
#***************************************************************************/

In [2]:
!pip3 install open-inference-openapi 
#!pip3 install httpx uvicorn

  Using cached open_inference_openapi-2.0.0.1-py3-none-any.whl.metadata (5.6 kB)
Using cached open_inference_openapi-2.0.0.1-py3-none-any.whl (29 kB)


### Deploy the Model to CAI Inference Service

In [3]:
#cdp iam generate-workload-auth-token --workload-name DE

/bin/bash: line 1: cdp: command not found


In [1]:
import os
myToken = "your-key"
modelId = "9198-lzx7-fuwj-cnhe"

In [2]:
import json
import httpx

#JWT = json.load(open("/tmp/jwt"))["access_token"]
JWT = myToken
headers = {'Authorization': 'Bearer ' + JWT,
           'Content-Type': 'application/json'}

httpx_client = httpx.Client(headers=headers)

In [4]:
CAII_DOMAIN_URL = 'https://inferenceservice.hl-dub-v.oldk-i9ly.a4.cloudera.site'

In [24]:
deploy_payload = {
 "namespace": "serving-default",
 "name": "fraud-model-onnx-test-31Mar-v3",
 "source": {
   "registry_source": {
      "model_id": modelId,
      "version": 2
    }
  },
 "resources": {
    "req_cpu": "2",
    "req_memory": "4Gi"
  },
  "api_standard": "oip",
  "has_chat_template": False,
  "metric_format": "triton",
  "task": "INFERENCE",
  "instance_type": "c4.4xlarge",
 "autoscaling": {
    "min_replicas": "1",
    "max_replicas": "2"
  }
}

r = httpx_client.post(CAII_DOMAIN_URL+'/api/v1alpha1/deployEndpoint', json=deploy_payload)
r

<Response [200 OK]>

In [7]:
# Wait for the endpoint to be ready
CAII_DOMAIN_URL = 'https://inferenceservice.hl-dub-v.oldk-i9ly.a4.cloudera.site'

describe_payload = {
    "namespace": "serving-default",
    "name": "fraud-model-onnx-test-31Mar-v3"
}

r = httpx_client.post(CAII_DOMAIN_URL+'/api/v1alpha1/describeEndpoint', json=describe_payload)
status = r.json()['status']
response = json.dumps(r.json(), indent=2)
print(response)

# Want target_model_state to get to 'Loaded'
status['target_model_state']

{
  "namespace": "serving-default",
  "name": "fraud-model-onnx-test-31Mar-v3",
  "url": "https://inferenceservice.hl-dub-v.oldk-i9ly.a4.cloudera.site/namespaces/serving-default/endpoints/fraud-model-onnx-test-31mar-v3/v2/models/9198-lzx7-fuwj-cnhe/infer",
  "conditions": [
    {
      "type": "IngressReady",
      "status": "True",
      "severity": "",
      "last_transition_time": "1774947277",
      "reason": "",
      "message": ""
    },
    {
      "type": "LatestDeploymentReady",
      "status": "True",
      "severity": "Info",
      "last_transition_time": "1774947277",
      "reason": "",
      "message": ""
    },
    {
      "type": "PredictorConfigurationReady",
      "status": "True",
      "severity": "Info",
      "last_transition_time": "1774947277",
      "reason": "",
      "message": ""
    },
    {
      "type": "PredictorReady",
      "status": "True",
      "severity": "",
      "last_transition_time": "1774947277",
      "reason": "",
      "message": ""
    },

'Loaded'

In [ ]:
#!pip3 install open-inference-openapi tritonclient[all]

In [8]:
# Get BASE_URL of the model endpoint. For predictive models, this is the endpoint URL preceding '/v2'
from urllib.parse import urlparse

r = httpx_client.post(CAII_DOMAIN_URL+'/api/v1alpha1/describeEndpoint', json=describe_payload)
url = r.json()['url']

# Remove last 4 path components to get the base url
parsed_url = urlparse(url)
path_parts = parsed_url.path.split("/")
new_path = "/".join(path_parts[:-4])
modified_url = parsed_url._replace(path=new_path).geturl()
modified_url

'https://inferenceservice.hl-dub-v.oldk-i9ly.a4.cloudera.site/namespaces/serving-default/endpoints/fraud-model-onnx-test-31mar-v3'

In [9]:
from open_inference.openapi.client import OpenInferenceClient, InferenceRequest
import httpx
import requests
import json

httpx_client = httpx.Client(headers=headers)
client = OpenInferenceClient(base_url=modified_url, httpx_client=httpx_client)
client.check_server_readiness()
metadata = client.read_model_metadata(modelId)
metadata_str = json.dumps(json.loads(metadata.json()), indent=2)
print(metadata_str)

{
  "name": "9198-lzx7-fuwj-cnhe",
  "versions": [
    "2"
  ],
  "platform": "onnxruntime_onnx",
  "inputs": [
    {
      "name": "input",
      "datatype": "FP32",
      "shape": [
        -1,
        15
      ]
    }
  ],
  "outputs": [
    {
      "name": "probabilities",
      "datatype": "FP32",
      "shape": [
        -1,
        2
      ]
    },
    {
      "name": "label",
      "datatype": "INT64",
      "shape": [
        -1,
        1
      ]
    }
  ]
}


In [10]:
import os, warnings, sys, logging
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score
import mlflow.sklearn
from xgboost import XGBClassifier
from datetime import date
import cml.data_v1 as cmldata
import pyspark.pandas as ps

USERNAME = os.environ["PROJECT_OWNER"]
DBNAME = os.environ["DBNAME_PREFIX"]+"_"+USERNAME
CONNECTION_NAME = os.environ["SPARK_CONNECTION_NAME"]

DATE = date.today()
EXPERIMENT_NAME = "xgb-cc-fraud-{0}".format(USERNAME)

mlflow.set_experiment(EXPERIMENT_NAME)

conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()

df_from_sql = ps.read_table('{0}.transactions_{1}'.format(DBNAME, USERNAME))
df = df_from_sql.to_pandas()
#df = df.drop(columns=["job"])

# test_size = 0.3
# X_train, X_test, y_train, y_test = train_test_split(df.drop("fraud_trx", axis=1), df["fraud_trx"], test_size=test_size)
y = df["fraud_trx"]
X = df.drop(columns=["fraud_trx"])
X.columns = ['f' + str(i) for i in range(len(X.columns))]

test_size = 0.3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

Setting spark.hadoop.yarn.resourcemanager.principal to user001


Spark Application Id:spark-application-1774948141063


Hive Session ID = 7b53c92d-83ff-4dce-866b-f360c77e948b
                                                                                

In [16]:
X_train.Co

,age,credit_card_balance,bank_account_balance,mortgage_balance,sec_bank_account_balance,savings_account_balance,sec_savings_account_balance,total_est_nworth,primary_loan_balance,secondary_loan_balance,uni_loan_balance,longitude,latitude,transaction_amount,customer_score
13541,63.0,942.0,83345.007812,555584.000000,66946.007812,135023.015625,478764.000000,21634.0,361.010010,227844.015625,770.010010,62.0,26.0,8238.009766,3.500179e+07
19657,16.0,27474.0,78544.007812,571524.000000,68069.007812,111090.007812,207784.015625,167043.0,4171.009766,206059.015625,8114.009766,127.0,-69.0,29633.009766,0.000000e+00
8320,14.0,2517.0,68466.007812,261479.015625,21946.009766,13919.009766,288951.000000,306656.0,2246.010010,43826.011719,6319.009766,-173.0,32.0,17308.009766,0.000000e+00
15761,50.0,13944.0,65363.011719,722203.000000,83927.007812,198231.015625,41031.011719,206840.0,2211.010010,35781.011719,3482.010010,149.0,-77.0,2144.010010,0.000000e+00
12291,40.0,18727.0,66794.007812,859135.000000,77670.007812,293124.000000,119545.007812,252573.0,2954.010010,150354.015625,2042.010010,158.0,-80.0,20568.009766,3.211607e-01


In [17]:
print(X_train.iloc[0].values.tolist())

[63.0, 942.0, 83345.0078125, 555584.0, 66946.0078125, 135023.015625, 478764.0, 21634.0, 361.010009765625, 227844.015625, 770.010009765625, 62.0, 26.0, 8238.009765625, 35001792.0]


In [18]:
import time

data = X_train.iloc[0].values.tolist()
payload = {
    "parameters": {
        "content_type": "pd"
    },
    "inputs": [
        {
            "name": "input",
            "datatype": "FP32",
            "shape": [1, 15],
            "data": data
        }
    ]
}
start = time.time()
pred = client.model_infer(
    modelId,
    request=InferenceRequest(
        inputs=payload["inputs"]
    ),
)

end = time.time()

json_resp_str = json.dumps(json.loads(pred.json()), indent=2)
print(json_resp_str)
print(f"latency={end-start}")

{
  "model_name": "9198-lzx7-fuwj-cnhe",
  "model_version": "2",
  "outputs": [
    {
      "name": "label",
      "shape": [
        1,
        1
      ],
      "datatype": "INT64",
      "data": [
        1.0
      ]
    },
    {
      "name": "probabilities",
      "shape": [
        1,
        2
      ],
      "datatype": "FP32",
      "data": [
        7.832050323486328e-05,
        0.9999216794967651
      ]
    }
  ]
}
latency=0.03600025177001953


In [19]:
#batch inferencing

In [21]:
import time
import json
import random
import math
from open_inference.openapi.client import InferenceRequest

# 1. Configuration
MAX_BATCH_SIZE = 4  # The limit defined by your endpoint error
num_total_records = random.randint(5, 20)
print(f"Processing {num_total_records} total records in chunks of {MAX_BATCH_SIZE}...")

# 2. Get the random sample from X_test
full_sample_df = X_test.sample(n=num_total_records)
num_features = full_sample_df.shape[1]

all_predictions = []
start_total = time.time()

# 3. Loop through the dataframe in chunks of 4
for i in range(0, num_total_records, MAX_BATCH_SIZE):
    # Slice the dataframe for the current batch
    batch_df = full_sample_df.iloc[i : i + MAX_BATCH_SIZE]
    current_batch_size = len(batch_df)
    
    # Flatten data for this specific batch
    batch_data = batch_df.values.flatten().tolist()
    
    print(f"  Sending batch {i//MAX_BATCH_SIZE + 1} ({current_batch_size} records)...")
    
    try:
        pred = client.model_infer(
            modelId,
            request=InferenceRequest(
                inputs=[{
                    "name": "input",
                    "datatype": "FP32",
                    "shape": [current_batch_size, num_features],
                    "data": batch_data
                }]
            ),
        )
        # Store the individual results
        all_predictions.append(json.loads(pred.json()))
        
    except Exception as e:
        print(f"  ❌ Error in batch starting at index {i}: {e}")

end_total = time.time()

# 4. Final Summary
print("-" * 30)
print(f"Successfully processed {len(all_predictions)} batches.")
print(f"Total time taken: {end_total - start_total:.4f}s")

# Optionally print the first result to verify
if all_predictions:
    print("\nFirst Batch Result Snippet:")
    print(json.dumps(all_predictions[0], indent=2))

Processing 16 total records in chunks of 4...
  Sending batch 1 (4 records)...
  Sending batch 2 (4 records)...
  Sending batch 3 (4 records)...
  Sending batch 4 (4 records)...
------------------------------
Successfully processed 4 batches.
Total time taken: 0.0537s

First Batch Result Snippet:
{
  "model_name": "9198-lzx7-fuwj-cnhe",
  "model_version": "2",
  "outputs": [
    {
      "name": "label",
      "shape": [
        4,
        1
      ],
      "datatype": "INT64",
      "data": [
        1.0,
        0.0,
        1.0,
        0.0
      ]
    },
    {
      "name": "probabilities",
      "shape": [
        4,
        2
      ],
      "datatype": "FP32",
      "data": [
        0.12430906295776367,
        0.8756909370422363,
        0.9996875524520874,
        0.00031244754791259766,
        0.0001628398895263672,
        0.9998371601104736,
        0.9997629523277283,
        0.00023704767227172852
      ]
    }
  ]
}
